In [2]:
pip install newspaper3k aiohttp aiofiles lxml nltk

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 58.9 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 8.5 MB/s eta 0:00:00
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13540 sha256=f0d56bd9ef5ec781693becffb1e82bd8983389db4d559578edde280bd2624de2
  Stored in directory: /root/.cache/pip/wheels/a5/91/9f/00d66475960891a64867914273fcaf78df6cb04d905b104a2a
  Created wheel for feedfinder2: filename=feedfinder2-0.0.4-py3-none-any.whl size=3341 sha256=00201828bb882f5a0d068ee39e9f86945e1f48978c913f344235c51e74744763
  Stored in directory: /root/.cache/pip/wheels/9f/9f/fb/364871d7426d3cdd4d293dcf

In [1]:
import os
import json
import asyncio
import aiohttp
import aiofiles
from bs4 import BeautifulSoup
from newspaper import Article # VŨ KHÍ TỐI THƯỢNG

# ==========================================
# CẤU HÌNH HỆ THỐNG V3.0 (SIÊU TỐC & SIÊU SẠCH)
# ==========================================
BASE_DIR = "/kaggle/working/data/raw"
TEXT_DIR = os.path.join(BASE_DIR, "texts")
IMAGE_DIR = os.path.join(BASE_DIR, "images")

os.makedirs(TEXT_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)

MAX_ARTICLES_PER_CATEGORY = 800 # Nâng trần lên 800 bài/nhãn để chống Overfitting
CONCURRENT_REQUESTS = 30 # Hạ xuống 30 luồng để CPU kịp xử lý phân tích chữ

RSS_FEEDS = {
    "World": ["https://vnexpress.net/rss/the-gioi.rss", "https://dantri.com.vn/rss/the-gioi.rss", "https://tuoitre.vn/rss/the-gioi.rss"],
    "Sports": ["https://vnexpress.net/rss/the-thao.rss", "https://dantri.com.vn/rss/the-thao.rss", "https://tuoitre.vn/rss/the-thao.rss"],
    "Business": ["https://vnexpress.net/rss/kinh-doanh.rss", "https://dantri.com.vn/rss/kinh-doanh.rss", "https://tuoitre.vn/rss/kinh-doanh.rss"],
    "Tech": ["https://vnexpress.net/rss/so-hoa.rss", "https://dantri.com.vn/rss/suc-manh-so.rss", "https://thanhnien.vn/rss/cong-nghe-game.rss"],
    "Health": ["https://vnexpress.net/rss/suc-khoe.rss", "https://dantri.com.vn/rss/suc-khoe.rss", "https://tuoitre.vn/rss/suc-khoe.rss"],
    "Science": ["https://vnexpress.net/rss/khoa-hoc.rss", "https://dantri.com.vn/rss/khoa-hoc-cong-nghe.rss", "https://tuoitre.vn/rss/khoa-hoc.rss"]
    # ... Bạn có thể thêm lại đầy đủ 12 chuyên mục như bản cũ ...
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"
}

# ==========================================
# HÀM BẤT ĐỒNG BỘ MỚI
# ==========================================
async def fetch_rss(session, rss_url):
    urls = []
    try:
        async with session.get(rss_url, headers=HEADERS, timeout=10) as response:
            xml_data = await response.text()
            soup = BeautifulSoup(xml_data, "xml")
            for item in soup.find_all("item"):
                link = item.find("link").text.strip()
                if "video" not in link and "podcast" not in link:
                    urls.append(link)
    except Exception:
        pass
    return urls

async def download_image_async(session, image_url, save_path):
    try:
        if image_url.startswith("//"): image_url = "https:" + image_url
        async with session.get(image_url, headers=HEADERS, timeout=10) as response:
            if response.status == 200:
                async with aiofiles.open(save_path, 'wb') as f:
                    await f.write(await response.read())
                return True
    except Exception:
        pass
    return False

async def scrape_article_smart(session, url, category, article_id, semaphore):
    """Cào bài báo kết hợp aiohttp (tốc độ mạng) và newspaper3k (thuật toán lọc lõi)"""
    async with semaphore:
        try:
            # 1. Tải HTML siêu tốc bằng aiohttp
            async with session.get(url, headers=HEADERS, timeout=10) as response:
                if response.status != 200: return False
                html = await response.text()

            # 2. Quăng HTML cho "Giáo sư" newspaper3k bóc tách
            article = Article(url, language='vi')
            article.set_html(html) # Ép nó đọc HTML mình vừa tải (không cho nó tự tải lại)
            article.parse() # Thuật toán Readability kích hoạt! Lọc sạch mọi quảng cáo, sidebar!

            title = article.title
            image_url = article.top_image
            content = article.text.strip()

            # Kiểm tra chất lượng (Chỉ lấy bài viết dài, có nội dung đàng hoàng)
            if not title or not image_url or len(content) < 300: 
                return False

            # Lấy 1000 ký tự đầu tiên để không làm tràn RAM của PhoBERT
            content_summary = content[:1000]

            # 3. Lưu xuống ổ cứng
            file_prefix = f"{category}_{article_id:04d}" 
            image_path = os.path.join(IMAGE_DIR, f"{file_prefix}.jpg")
            
            if await download_image_async(session, image_url, image_path):
                text_data = {
                    "id": file_prefix,
                    "url": url,
                    "category": category,
                    "title": title,
                    "content": content_summary,
                    "image_file": f"{file_prefix}.jpg"
                }
                text_path = os.path.join(TEXT_DIR, f"{file_prefix}.json")
                async with aiofiles.open(text_path, "w", encoding="utf-8") as f:
                    await f.write(json.dumps(text_data, ensure_ascii=False, indent=4))
                return True

        except Exception:
            pass
        return False

async def main():
    print(f"🚀 BẮT ĐẦU CÀO DATA V3.0 (NEWSPAPER3K SMART EXTRACTION)...")
    semaphore = asyncio.Semaphore(CONCURRENT_REQUESTS)
    total_saved = 0

    async with aiohttp.ClientSession() as session:
        for category, rss_url_list in RSS_FEEDS.items():
            print(f"\n📡 Đang quét chuyên mục: [{category}]")
            
            rss_tasks = [fetch_rss(session, rss_url) for rss_url in rss_url_list]
            rss_results = await asyncio.gather(*rss_tasks)
            urls = list(set([url for sublist in rss_results for url in sublist]))
            
            print(f"🔗 Tìm thấy {len(urls)} links. Bắt đầu dùng AI bóc tách lõi văn bản...")

            scrape_tasks = []
            for i, url in enumerate(urls[:MAX_ARTICLES_PER_CATEGORY * 2]): 
                scrape_tasks.append(scrape_article_smart(session, url, category, i+1, semaphore))

            results = await asyncio.gather(*scrape_tasks)
            saved = sum(1 for r in results if r)
            total_saved += saved
            print(f"✅ Hoàn thành [{category}]. Thu được: {saved} bài báo SIÊU SẠCH (Đã lọc Sidebar/QC).")

    print("-" * 60)
    print(f"🎉 TỔNG KẾT: Thu thập thành công {total_saved} bài báo chất lượng cao!")

# Chạy!
await main()

ModuleNotFoundError: No module named 'newspaper'

In [2]:
!pip install transformers requests beautifulsoup4

In [4]:
import os
import json
import random
import logging
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models

# "Vũ khí hạng nặng" từ Hugging Face
from transformers import AutoModel, AutoTokenizer

# ==========================================
# 0. CẤU HÌNH & MLOPS TOOLS
# ==========================================
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
logger = logging.getLogger(__name__)

BASE_DIR = "/kaggle/working"
RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")
MODEL_SAVE_PATH = os.path.join(BASE_DIR, "transformer_model.pth") # Đích đến của model tốt nhất

# HYPERPARAMETERS DÀNH RIÊNG CHO FINE-TUNING
BATCH_SIZE = 16       # VRAM Kaggle GPU T4 là 15GB, để 16 là vừa đẹp không bị tràn RAM
MAX_EPOCHS = 10       # Transformer học rất nhanh, thường 3-4 Epoch là hội tụ
LEARNING_RATE = 2e-5  # TUYỆT ĐỐI KHÔNG ĐỂ 1e-3. 2e-5 là tiêu chuẩn Fine-tune!
MAX_SEQ_LEN = 256     # Giới hạn chiều dài câu để cân bằng RAM và Ngữ nghĩa

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): 
        torch.cuda.manual_seed_all(seed)

class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.005):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            logger.warning(f"⚠️ Early Stopping Counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

# ==========================================
# 1. BỘ DỮ LIỆU ĐA PHƯƠNG THỨC (DATASET)
# ==========================================
class TransformerNewsDataset(Dataset):
    def __init__(self, data_dir, transform, tokenizer, max_len=256):
        self.text_dir = os.path.join(data_dir, "texts")
        self.image_dir = os.path.join(data_dir, "images")
        self.transform = transform
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        self.samples = []
        self.labels = []
        
        # Tự động trích xuất các nhãn hiện có
        self.classes = sorted(list(set([f.split('_')[0] for f in os.listdir(self.text_dir) if f.endswith('.json')])))
        self.category_to_id = {cat: idx for idx, cat in enumerate(self.classes)}
        self.id_to_category = {idx: cat for cat, idx in self.category_to_id.items()}

        for filename in os.listdir(self.text_dir):
            if filename.endswith(".json"):
                base_name = filename.replace(".json", "")
                img_path = os.path.join(self.image_dir, f"{base_name}.jpg")
                if os.path.exists(img_path):
                    with open(os.path.join(self.text_dir, filename), 'r', encoding='utf-8') as f:
                        cat = json.load(f)['category']
                    self.samples.append((os.path.join(self.text_dir, filename), img_path))
                    self.labels.append(self.category_to_id[cat])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text_path, img_path = self.samples[idx]
        
        # Xử lý Ảnh
        image = Image.open(img_path).convert("RGB")
        image_tensor = self.transform(image)
        
        # Xử lý Text bằng Tokenizer thần thánh
        with open(text_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        full_text = f"{data.get('title', '')}. {data.get('content', '')}"
        
        # GỌI TRỰC TIẾP TOKENIZER NHƯ MỘT HÀM (Callable)
        encoding = self.tokenizer(
            full_text,
            add_special_tokens=True, 
            max_length=self.max_len,
            padding='max_length',    
            truncation=True,         
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        label_id = self.category_to_id[data['category']]
        
        return {
            'image': image_tensor,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label_id, dtype=torch.long)
        }

# ==========================================
# 2. KIẾN TRÚC FUSION (PHOBERT + RESNET)
# ==========================================
class PhoBertResNetFusion(nn.Module):
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        
        # --- VISION: ResNet50 ---
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.image_encoder = nn.Sequential(*list(resnet.children())[:-1]) # Chặt đầu FC, lấy vector 2048
        
        # --- NLP: PhoBERT Base V2 ---
        self.text_encoder = AutoModel.from_pretrained("vinai/phobert-base-v2") # Vector [CLS] 768
        
        # --- FUSION: Hợp nhất ---
        fusion_dim = 2048 + 768 
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, images, input_ids, attention_mask):
        # Đặc trưng Ảnh
        img_features = torch.flatten(self.image_encoder(images), 1) 
        
        # Đặc trưng Text (Lấy pooler_output - Tóm tắt của cả câu)
        text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_features = text_outputs.pooler_output 
        
        # Phân loại
        combined = torch.cat((img_features, text_features), dim=1) 
        return self.classifier(combined)

# ==========================================
# 3. VÒNG LẶP HUẤN LUYỆN (MAIN)
# ==========================================
def main():
    set_seed()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"🚀 Khởi động Fine-Tuning Transformer trên: {device.type.upper()}")

    # 1. Tokenizer & Transforms
    logger.info("Tải PhoBERT Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5), # Augmentation
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    full_train_ds = TransformerNewsDataset(RAW_DATA_DIR, train_transform, tokenizer, MAX_SEQ_LEN)
    full_val_ds = TransformerNewsDataset(RAW_DATA_DIR, val_transform, tokenizer, MAX_SEQ_LEN)
    
    # 2. Stratified Split (Chia Data)
    indices, labels = list(range(len(full_train_ds))), full_train_ds.labels
    train_idx, temp_idx, _, temp_labels = train_test_split(indices, labels, test_size=0.3, stratify=labels, random_state=42)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=temp_labels, random_state=42)
    
    train_loader = DataLoader(Subset(full_train_ds, train_idx), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(Subset(full_val_ds, val_idx), batch_size=BATCH_SIZE)
    test_loader = DataLoader(Subset(full_val_ds, test_idx), batch_size=BATCH_SIZE)

    # 3. Khởi tạo Mô hình & Tối ưu hóa
    logger.info("Khởi tạo mạng PhoBert + ResNet50...")
    model = PhoBertResNetFusion(num_classes=len(full_train_ds.classes)).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4) # AdamW cho Transformer
    early_stopping = EarlyStopping(patience=3)

    # 4. Huấn luyện
    logger.info("🔥 BẮT ĐẦU FINE-TUNING 🔥")
    best_val_loss = float("inf")

    for epoch in range(MAX_EPOCHS):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        for batch in train_loader:
            imgs = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)
            masks = batch['attention_mask'].to(device)
            lbls = batch['label'].to(device)

            optimizer.zero_grad()
            logits = model(imgs, input_ids, masks)
            loss = criterion(logits, lbls)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_correct += (logits.argmax(1) == lbls).sum().item()
            train_total += lbls.size(0)

        # Đánh giá (Validation)
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for batch in val_loader:
                imgs = batch['image'].to(device)
                input_ids = batch['input_ids'].to(device)
                masks = batch['attention_mask'].to(device)
                lbls = batch['label'].to(device)

                logits = model(imgs, input_ids, masks)
                val_loss += criterion(logits, lbls).item()
                val_correct += (logits.argmax(1) == lbls).sum().item()
                val_total += lbls.size(0)

        avg_val_loss = val_loss / len(val_loader)
        logger.info(
            f"Epoch {epoch+1:02d} | Train Acc: {train_correct/train_total*100:.2f}% "
            f"| Val Acc: {val_correct/val_total*100:.2f}% | Val Loss: {avg_val_loss:.4f}"
        )

        early_stopping(avg_val_loss)
        if early_stopping.counter == 0:
            torch.save({
                "model_state_dict": model.state_dict(),
                "category_to_id": full_train_ds.category_to_id,
                "id_to_category": full_train_ds.id_to_category,
                "max_seq_len": MAX_SEQ_LEN
            }, MODEL_SAVE_PATH)
            logger.info("   🌟 Đã lưu File Trọng số (Model tốt nhất hiện tại)!")
            
        if early_stopping.early_stop:
            logger.info("🛑 Mạng đã hội tụ. Kích hoạt Early Stopping!")
            break

    # 5. Bài thi cuối kỳ
    logger.info("🚀 BÀI THI CUỐI KỲ TRÊN TẬP TEST 🚀")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH)["model_state_dict"])
    model.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            imgs, input_ids, masks, lbls = batch['image'].to(device), batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['label'].to(device)
            test_correct += (model(imgs, input_ids, masks).argmax(1) == lbls).sum().item()
            test_total += lbls.size(0)
            
    logger.info(f"🎯 KẾT QUẢ ĐỘ CHÍNH XÁC PHO-BERT + RESNET: {test_correct/test_total*100:.2f}%")

if __name__ == "__main__":
    main()

2026-03-13 11:36:16,576 - 🚀 Khởi động Fine-Tuning Transformer trên: CUDA
2026-03-13 11:36:16,577 - Tải PhoBERT Tokenizer...
2026-03-13 11:36:16,718 - HTTP Request: HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-13 11:36:16,733 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/vinai/phobert-base-v2/e2375d266bdf39c6e8e9a87af16a5da3190b0cc8/config.json "HTTP/1.1 200 OK"
2026-03-13 11:36:16,795 - HTTP Request: HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-03-13 11:36:16,860 - HTTP Request: HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-03-13 11:36:16,924 - HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-base-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-13 11:36:16,991 - HTTP Request: GET https:/

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

2026-03-13 11:36:18,981 - HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-base-v2/commits/main "HTTP/1.1 200 OK"
RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
2026-03-13 11:36:19,103 - HTTP Request: GET https://huggingface.co/api/models/vinai/phob

In [28]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer
from torchvision import transforms
from PIL import Image
import requests
from bs4 import BeautifulSoup
from io import BytesIO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_SAVE_PATH = "/kaggle/working/transformer_model.pth"

# 1. Khôi phục trạng thái
checkpoint = torch.load(MODEL_SAVE_PATH)
id_to_category = checkpoint['id_to_category']
num_classes = len(id_to_category)

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
model = PhoBertResNetFusion(num_classes=num_classes).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


def scrape_any_news_url(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(response.content, "html.parser")
        
        # 1. LẤY TIÊU ĐỀ (Cách an toàn nhất: Dùng thẻ Meta OG)
        # Báo nào cũng phải có thẻ này để share lên Facebook
        title_meta = soup.find("meta", property="og:title")
        if title_meta and title_meta.get("content"):
            title = title_meta["content"].strip()
        else:
            # Dự phòng: tìm thẻ h1 hoặc title
            h1 = soup.find("h1")
            title = h1.text.strip() if h1 else soup.find("title").text.strip()
            
        # 2. LẤY ẢNH (Dùng Meta OG - Chuẩn tuyệt đối)
        img_meta = soup.find("meta", property="og:image")
        image_url = img_meta["content"] if img_meta else ""
        if image_url.startswith("//"): image_url = "https:" + image_url
            
        # 3. KHOANH VÙNG NỘI DUNG CHÍNH (Xử lý mọi biến thể của VnExpress)
        # Bổ sung thêm vùng chứa cho bài Video và Bài Ảnh
        article_body = soup.find("article", class_="fck_detail") # VnExpress
        if not article_body: article_body = soup.find("div", class_="video-detail") # VnExpress Video
        if not article_body: article_body = soup.find("div", class_="singular-content") # Dân Trí
        if not article_body: article_body = soup.find("div", class_="detail-content") # Tuổi Trẻ, Thanh Niên
        if not article_body: article_body = soup.find("div", class_="post-content") # BÁO CÔNG AN NHÂN DÂN
        if not article_body: article_body = soup.find("div", class_="article-content") # Phổ biến chung
        if not article_body: article_body = soup.find("article") # Fallback cuối cùng
        
        # 4. TRÍCH XUẤT ĐOẠN VĂN
        if article_body:
            paragraphs = article_body.find_all("p")
        else:
            # Cực kỳ quan trọng: Nếu không tìm thấy vùng nội dung chính, 
            # TUYỆT ĐỐI KHÔNG dùng soup.find_all("p") để tránh cào nhầm rác ở sidebar.
            # Trả về lỗi luôn.
            return None, None, f"Không tìm thấy cấu trúc nội dung của bài báo này."
            
        # Lấy 10 đoạn đầu tiên, mỗi đoạn dài hơn 30 ký tự (bài video thường chữ ngắn)
        content_parts = [p.get_text().strip() for p in paragraphs if len(p.get_text().strip()) > 30]
        content = " ".join(content_parts[:10])
        
        # Bài video có thể không có text, ta lấy luôn tiêu đề làm nội dung để dự đoán
        if not content:
            content = title
            
        if not image_url: 
            return None, None, f"Báo này thiếu thẻ Ảnh (og:image)"
            
        img_response = requests.get(image_url, headers=headers)
        image = Image.open(BytesIO(img_response.content)).convert("RGB")
        return image, f"{title}. {content}", title
        
    except Exception as e: 
        return None, None, f"Lỗi Python: {str(e)}"

def predict_transformer(url):
    print(f"\n🌍 Đang phân tích: {url}")
    image, full_text, title = scrape_any_news_url(url)
    
    if not image: 
        # IN CHI TIẾT LỖI RA MÀN HÌNH ĐỂ DEBUG
        print(f"❌ Lỗi cào dữ liệu chi tiết: {title}")
        return
        
    # Chuẩn bị Tensor Ảnh
    img_tensor = val_transform(image).unsqueeze(0).to(device)
    
    # Chuẩn bị Tensor Text (Đã xóa chữ self.)
    encoding = tokenizer(
        full_text, max_length=256, padding='max_length', truncation=True, return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)
    masks = encoding['attention_mask'].to(device)
    
    # Tiên đoán
    with torch.no_grad():
        outputs = model(img_tensor, input_ids, masks)
        probabilities = F.softmax(outputs, dim=1).squeeze()
        
    top3_prob, top3_idx = torch.topk(probabilities, 3)
    
    print(f"📰 Tiêu đề: {title}")
    print("🔮 KẾT QUẢ TỪ PHOBERT + RESNET:")
    print("-" * 40)
    for i in range(3):
        print(f"   Top {i+1}: [{id_to_category[top3_idx[i].item()]}] - {top3_prob[i].item()*100:.2f}%")
    print("-" * 40)

# ---- TEST THỰC TẾ ----
# test_url = "https://vnexpress.net/bong-cuoi-tan-pha-he-than-kinh-the-nao-3891404.html"

# Hoặc một bài test độ hóc búa (Bóng đá nhưng có chữ "bóng"):
test_url = "https://baomoi.com/tro-vui-doi-giong-tu-bong-heli-va-nhung-rui-ro-it-nguoi-biet-c54679594.epi"

predict_transformer(test_url)

2026-03-13 13:14:58,325 - HTTP Request: HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-13 13:14:58,341 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/vinai/phobert-base-v2/e2375d266bdf39c6e8e9a87af16a5da3190b0cc8/config.json "HTTP/1.1 200 OK"
2026-03-13 13:14:58,407 - HTTP Request: HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-03-13 13:14:58,470 - HTTP Request: HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-03-13 13:14:58,531 - HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-base-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-13 13:14:58,597 - HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-base-v2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-03-13 13:14:58

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

2026-03-13 13:14:59,617 - HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-base-v2/commits/main "HTTP/1.1 200 OK"
RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
2026-03-13 13:14:59,732 - HTTP Request: GET https://huggingface.co/api/models/vinai/phob


🌍 Đang phân tích: https://baomoi.com/tro-vui-doi-giong-tu-bong-heli-va-nhung-rui-ro-it-nguoi-biet-c54679594.epi
❌ Lỗi cào dữ liệu chi tiết: Không tìm thấy cấu trúc nội dung của bài báo này.


In [15]:
import os
import json
import random
import logging
import numpy as np
from collections import Counter
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

# ==========================================
# 0. CẤU HÌNH & EARLY STOPPING
# ==========================================
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

BASE_DIR = "/kaggle/working"
RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")
MODEL_SAVE_PATH = os.path.join(BASE_DIR, "best_baseline_model.pth")

BATCH_SIZE = 32
MAX_EPOCHS = 50       # Tăng vọt lên 50 Epochs, đã có Early Stopping lo!
LEARNING_RATE = 1e-3
MAX_VOCAB_SIZE = 30000 # Tăng vocab size lên để bao quát từ vựng 12 nhãn

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class EarlyStopping:
    """Công cụ 'canh gác' tránh Overfitting"""
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            logger.info(f"⚠️ Early Stopping Counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

# ==========================================
# 1. PHÂN TÍCH DỮ LIỆU & TIỀN XỬ LÝ
# ==========================================
def analyze_corpus(text_dir):
    """Đọc toàn bộ văn bản để tìm ra max_seq_len khoa học nhất"""
    logger.info("Đang phân tích độ dài các bài báo...")
    lengths = []
    all_texts = []
    
    for f_name in os.listdir(text_dir):
        if f_name.endswith(".json"):
            with open(os.path.join(text_dir, f_name), 'r', encoding='utf-8') as f:
                d = json.load(f)
                full_text = f"{d.get('title', '')}. {d.get('content', '')}"
                words = full_text.lower().split()
                lengths.append(len(words))
                all_texts.append(full_text)
                
    mean_len = np.mean(lengths)
    p90 = np.percentile(lengths, 90)
    p95 = np.percentile(lengths, 95)
    
    logger.info(f"📊 Thống kê Corpus: Trung bình {mean_len:.0f} từ/bài.")
    logger.info(f"📊 90% bài báo có độ dài < {p90:.0f} từ.")
    logger.info(f"📊 95% bài báo có độ dài < {p95:.0f} từ.")
    
    # Chọn Phân vị thứ 95 làm chuẩn để cắt cụt
    optimal_max_len = int(p95)
    return optimal_max_len, all_texts

class TextProcessor:
    def __init__(self, max_vocab_size, max_seq_len):
        self.max_vocab_size = max_vocab_size
        self.max_seq_len = max_seq_len
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}
        self.idx2word = {0: "<PAD>", 1: "<UNK>"}

    def clean_text(self, text):
        import re
        text = text.lower()
        text = re.sub(r'[^\w\s]', ' ', text)
        return text.split()

    def build_vocab(self, texts):
        all_words = []
        for text in texts:
            all_words.extend(self.clean_text(text))
        
        word_counts = Counter(all_words)
        common_words = word_counts.most_common(self.max_vocab_size - 2)
        
        for idx, (word, _) in enumerate(common_words, start=2):
            self.word2idx[word] = idx
            self.idx2word[idx] = word

    def text_to_tensor(self, text):
        words = self.clean_text(text)
        indices = [self.word2idx.get(w, self.word2idx["<UNK>"]) for w in words]
        
        seq_len = min(len(indices), self.max_seq_len)
        if len(indices) < self.max_seq_len:
            indices += [self.word2idx["<PAD>"]] * (self.max_seq_len - len(indices))
        else:
            indices = indices[:self.max_seq_len]
            
        return torch.tensor(indices, dtype=torch.long), torch.tensor(seq_len, dtype=torch.long)

class NewsDataset(Dataset):
    def __init__(self, data_dir, transform, text_processor):
        self.text_dir = os.path.join(data_dir, "texts")
        self.image_dir = os.path.join(data_dir, "images")
        self.transform = transform
        self.text_processor = text_processor
        
        self.samples = []
        self.labels = [] # Lưu lại nhãn để dùng cho Stratified Split
        
        self.classes = sorted(list(set([f.split('_')[0] for f in os.listdir(self.text_dir) if f.endswith('.json')])))
        self.category_to_id = {cat: idx for idx, cat in enumerate(self.classes)}

        for filename in os.listdir(self.text_dir):
            if filename.endswith(".json"):
                base_name = filename.replace(".json", "")
                img_path = os.path.join(self.image_dir, f"{base_name}.jpg")
                if os.path.exists(img_path):
                    with open(os.path.join(self.text_dir, filename), 'r', encoding='utf-8') as f:
                        cat = json.load(f)['category']
                    self.samples.append((os.path.join(self.text_dir, filename), img_path))
                    self.labels.append(self.category_to_id[cat])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text_path, img_path = self.samples[idx]
        image = self.transform(Image.open(img_path).convert("RGB"))
        
        with open(text_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        full_text = f"{data.get('title', '')}. {data.get('content', '')}"
        
        text_tensor, text_length = self.text_processor.text_to_tensor(full_text)
        label_id = self.category_to_id[data['category']]
        
        return image, text_tensor, text_length, torch.tensor(label_id, dtype=torch.long)

# ==========================================
# 2. MÔ HÌNH CNN + BiLSTM TỪ ĐẦU (Như cũ)
# ==========================================
class NewsImageExtractor(nn.Module):
    def __init__(self, output_dim=512, dropout=0.3):
        super().__init__()
        self.output_dim = output_dim
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.projection = nn.Sequential(nn.Dropout(dropout), nn.Linear(256, output_dim), nn.ReLU())

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.projection(x)

class GlobalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_outputs, mask):
        attn_weights = self.attention(lstm_outputs).squeeze(2)
        attn_weights = attn_weights.masked_fill(mask == False, -1e9)
        attn_weights = F.softmax(attn_weights, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), lstm_outputs).squeeze(1)
        return context

class NewsTextExtractor(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers = 2, batch_first=True, bidirectional=True, dropout=dropout)
        self.attention = GlobalAttention(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = hidden_dim * 2

    def forward(self, text_tensor, text_lengths):
        mask = text_tensor != 0
        embedded = self.dropout(self.embedding(text_tensor))
        lengths_cpu = text_lengths.cpu() if text_lengths.is_cuda else text_lengths
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths_cpu, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=text_tensor.size(1))
        return self.attention(lstm_out, mask)

class MultimodalNewsClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, dropout=0.4): # Tăng dropout lên chút xíu chống overfit
        super().__init__()
        self.image_extractor = NewsImageExtractor(output_dim=512, dropout=dropout)
        self.text_extractor = NewsTextExtractor(vocab_size=vocab_size, hidden_dim=256, dropout=dropout)
        
        fusion_dim = 1024
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, images, text_tensor, text_lengths):
        img_vec = self.image_extractor(images)
        txt_vec = self.text_extractor(text_tensor, text_lengths)
        combined = torch.cat((img_vec, txt_vec), dim=1)
        return self.classifier(combined)

# ==========================================
# 3. QUY TRÌNH HUẤN LUYỆN (TÍCH HỢP STRATIFIED)
# ==========================================
def main():
    set_seed()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Khảo sát & Build Vocab
    text_dir = os.path.join(RAW_DATA_DIR, "texts")
    optimal_max_len, all_texts = analyze_corpus(text_dir)
    
    processor = TextProcessor(max_vocab_size=MAX_VOCAB_SIZE, max_seq_len=optimal_max_len)
    processor.build_vocab(all_texts)
    
   # ==========================================
    # KHỞI TẠO 2 BỘ TRANSFORM RIÊNG BIỆT
    # ==========================================
    # 1. Transform Tăng cường (Chỉ cho Train)
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5), # Lật ngang
        transforms.RandomRotation(15),          # Xoay tối đa 15 độ
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    # 2. Transform Tiêu chuẩn (Chỉ cho Val và Test)
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    # Khởi tạo 2 bản sao Dataset song song (Do đọc từ file nên rất nhẹ, không tốn RAM)
    full_train_dataset = NewsDataset(RAW_DATA_DIR, train_transform, processor)
    full_val_dataset = NewsDataset(RAW_DATA_DIR, val_transform, processor)
    
    # Lấy danh sách index và label để chia Stratified
    indices = list(range(len(full_train_dataset)))
    labels = full_train_dataset.labels
    
    # ==========================================
    # STRATIFIED SPLIT (GÁN INDEX VÀO ĐÚNG BẢN SAO DATASET)
    # ==========================================
    train_idx, temp_idx, _, temp_labels = train_test_split(
        indices, labels, test_size=0.3, stratify=labels, random_state=42
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5, stratify=temp_labels, random_state=42
    )
    
    logger.info(f"Đã chia Stratified: Train={len(train_idx)}, Val={len(val_idx)}, Test={len(test_idx)}")
    
    # CHIÊU CHUẨN XÁC: Train dùng dataset có transform lật/xoay, Val/Test dùng dataset nguyên bản
    train_ds = Subset(full_train_dataset, train_idx)
    val_ds = Subset(full_val_dataset, val_idx)
    test_ds = Subset(full_val_dataset, test_idx)
    
    # Nạp vào DataLoader
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

    # 3. Model & Optim
    model = MultimodalNewsClassifier(len(processor.word2idx), len(full_train_dataset.classes)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay = 1e-4)
    # Lắp hộp số tự động
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
    )
    # Khởi tạo Early Stopping (Chịu đựng tối đa 5 Epochs không tiến bộ)
    early_stopping = EarlyStopping(patience=5, min_delta=0.005)

    # 4. Train Loop
    logger.info(f"🔥 BẮT ĐẦU HUẤN LUYỆN: Tối đa {MAX_EPOCHS} Epochs (Có Early Stopping) 🔥")

    for epoch in range(MAX_EPOCHS):
        model.train()
        t_loss, t_corr, t_tot = 0, 0, 0
        
        for imgs, txts, lens, lbls in train_loader:
            imgs, txts, lens, lbls = imgs.to(device), txts.to(device), lens.to(device), lbls.to(device)
            optimizer.zero_grad()
            out = model(imgs, txts, lens)
            loss = criterion(out, lbls)
            loss.backward()
            optimizer.step()
            
            t_loss += loss.item()
            t_corr += (out.argmax(1) == lbls).sum().item()
            t_tot += lbls.size(0)

        # Eval
        model.eval()
        v_loss, v_corr, v_tot = 0, 0, 0
        with torch.no_grad():
            for imgs, txts, lens, lbls in val_loader:
                imgs, txts, lens, lbls = imgs.to(device), txts.to(device), lens.to(device), lbls.to(device)
                out = model(imgs, txts, lens)
                v_loss += criterion(out, lbls).item()
                v_corr += (out.argmax(1) == lbls).sum().item()
                v_tot += lbls.size(0)

        avg_val_loss = v_loss/len(val_loader)
        logger.info(f"Epoch {epoch+1:02d} | Train Acc: {t_corr/t_tot*100:.2f}% | Val Acc: {v_corr/v_tot*100:.2f}% | Val Loss: {avg_val_loss:.4f}")

        # Gửi Val Loss cho lính canh Early Stopping
        early_stopping(avg_val_loss)
        # Giảm LR nếu Val Loss đi ngang
        scheduler.step(avg_val_loss)
        
        if early_stopping.counter == 0: # Đồng nghĩa với việc tìm ra Val Loss tốt nhất mới
            torch.save({'model_state_dict': model.state_dict()}, MODEL_SAVE_PATH)
            logger.info(" -> 🌟 Đã lưu Checkpoint mới!")
            
        if early_stopping.early_stop:
            logger.info("🛑 EARLY STOPPING KÍCH HOẠT! Mô hình đã bắt đầu Overfit. Dừng Train!")
            break

    # 5. Bài thi Test cuối cùng
    model.load_state_dict(torch.load(MODEL_SAVE_PATH)['model_state_dict'])
    model.eval()
    test_corr, test_tot = 0, 0
    with torch.no_grad():
        for imgs, txts, lens, lbls in test_loader:
            imgs, txts, lens, lbls = imgs.to(device), txts.to(device), lens.to(device), lbls.to(device)
            test_corr += (model(imgs, txts, lens).argmax(1) == lbls).sum().item()
            test_tot += lbls.size(0)
            
    logger.info(f"🎯 KẾT QUẢ TEST CUỐI CÙNG: {test_corr/test_tot*100:.2f}%")

if __name__ == "__main__":
    main()

2026-03-13 05:15:40,238 - INFO - Đang phân tích độ dài các bài báo...
2026-03-13 05:15:41,181 - INFO - 📊 Thống kê Corpus: Trung bình 466 từ/bài.
2026-03-13 05:15:41,182 - INFO - 📊 90% bài báo có độ dài < 652 từ.
2026-03-13 05:15:41,183 - INFO - 📊 95% bài báo có độ dài < 723 từ.
2026-03-13 05:15:43,561 - INFO - Đã chia Stratified: Train=4977, Val=1066, Test=1067
2026-03-13 05:15:43,621 - INFO - 🔥 BẮT ĐẦU HUẤN LUYỆN: Tối đa 50 Epochs (Có Early Stopping) 🔥
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
2026-03-13 05:18:35,054 - INFO - Epoch 01 | Train Acc: 65.97% | Val Acc: 79.74% | Val Loss: 0.6842
2026-03-13 05:18:35,108 - INFO -  -> 🌟 Đã lưu Checkpoint mới!
2026-03-13 05:21:27,765 - INFO - Epoch 02 | Train Acc: 84.05% | Val Acc: 85.74% | Val Loss: 0.4891
2026-03-13 05:21:27,838 - INFO -  -> 🌟 Đã lưu Checkpoint mới!
2026-03-13 05:24:24,221 - INFO - Epoch 03 | 

In [5]:
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import os
from bs4 import BeautifulSoup
from PIL import Image
from io import BytesIO
from torchvision import transforms
from collections import Counter

# ==========================================
# 1. CẤU HÌNH & LOAD LẠI TỪ ĐIỂN (VOCAB)
# ==========================================
BASE_DIR = "/kaggle/working"
MODEL_SAVE_PATH = os.path.join(BASE_DIR, "base_line_model.pth")
RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🤖 Đang khởi động AI trên thiết bị: {device.type.upper()}")

# Hàm khôi phục lại Vocab và Danh sách Nhãn từ thư mục data
def restore_vocab_and_classes(data_dir, max_vocab_size=30000, max_seq_len=668): # max_seq_len lấy từ log của bạn
    text_dir = os.path.join(data_dir, "texts")
    classes = sorted(list(set([f.split('_')[0] for f in os.listdir(text_dir) if f.endswith('.json')])))
    id_to_category = {idx: cat for idx, cat in enumerate(classes)}
    
    word2idx = {"<PAD>": 0, "<UNK>": 1}
    all_words = []
    import re
    for f_name in os.listdir(text_dir):
        if f_name.endswith(".json"):
            with open(os.path.join(text_dir, f_name), 'r', encoding='utf-8') as f:
                d = json.load(f)
                text = f"{d.get('title', '')}. {d.get('content', '')}".lower()
                text = re.sub(r'[^\w\s]', ' ', text)
                all_words.extend(text.split())
                
    word_counts = Counter(all_words)
    for idx, (word, _) in enumerate(word_counts.most_common(max_vocab_size - 2), start=2):
        word2idx[word] = idx
        
    return word2idx, id_to_category, max_seq_len

word2idx, id_to_category, MAX_SEQ_LEN = restore_vocab_and_classes(RAW_DATA_DIR)
vocab_size = len(word2idx)
num_classes = len(id_to_category)

# ==========================================
# 2. ĐỊNH NGHĨA LẠI MODEL ĐỂ LOAD TRỌNG SỐ
# ==========================================
class NewsImageExtractor(nn.Module):
    def __init__(self, output_dim=512, dropout=0.3):
        super().__init__()
        self.output_dim = output_dim
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.projection = nn.Sequential(nn.Dropout(dropout), nn.Linear(256, output_dim), nn.ReLU())

    def forward(self, x):
        return self.projection(torch.flatten(self.global_pool(self.features(x)), 1))

class GlobalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_outputs, mask):
        attn_weights = F.softmax(self.attention(lstm_outputs).squeeze(2).masked_fill(mask == False, -1e9), dim=1)
        return torch.bmm(attn_weights.unsqueeze(1), lstm_outputs).squeeze(1)

class NewsTextExtractor(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2, batch_first=True, bidirectional=True, dropout=dropout)
        self.attention = GlobalAttention(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = hidden_dim * 2

    def forward(self, text_tensor, text_lengths):
        mask = text_tensor != 0
        packed = nn.utils.rnn.pack_padded_sequence(self.dropout(self.embedding(text_tensor)), text_lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=text_tensor.size(1))
        return self.attention(lstm_out, mask)

class MultimodalNewsClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, dropout=0.4):
        super().__init__()
        self.image_extractor = NewsImageExtractor(dropout=dropout)
        self.text_extractor = NewsTextExtractor(vocab_size=vocab_size, dropout=dropout)
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(512, num_classes)
        )

    def forward(self, images, text_tensor, text_lengths):
        return self.classifier(torch.cat((self.image_extractor(images), self.text_extractor(text_tensor, text_lengths)), dim=1))

# Khởi tạo và Load Model
model = MultimodalNewsClassifier(vocab_size, num_classes).to(device)
model.load_state_dict(torch.load(MODEL_SAVE_PATH)['model_state_dict'])
model.eval()
print("✅ Đã load thành công siêu mô hình Đa phương thức (Độ chính xác 90.91%)!")

# ==========================================
# 3. CÁC HÀM TIỀN XỬ LÝ (MINI-PIPELINE)
# ==========================================
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def process_text_for_prediction(text):
    import re
    words = re.sub(r'[^\w\s]', ' ', text.lower()).split()
    indices = [word2idx.get(w, word2idx["<UNK>"]) for w in words]
    seq_len = min(len(indices), MAX_SEQ_LEN)
    if len(indices) < MAX_SEQ_LEN:
        indices += [word2idx["<PAD>"]] * (MAX_SEQ_LEN - len(indices))
    else:
        indices = indices[:MAX_SEQ_LEN]
    return torch.tensor([indices], dtype=torch.long), torch.tensor([seq_len], dtype=torch.long)

def scrape_any_news_url(url):
    """Cào cấu nội dung từ một link báo bất kỳ (Zing, CafeF, Kenh14...)"""
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, "html.parser")
        
        title_meta = soup.find("meta", property="og:title")
        title = title_meta["content"].strip() if title_meta else soup.find("title").text
        
        img_meta = soup.find("meta", property="og:image")
        image_url = img_meta["content"] if img_meta else ""
        
        paragraphs = soup.find_all("p")
        content = " ".join([p.get_text().strip() for p in paragraphs if len(p.get_text().strip()) > 50][:10])
        
        if not image_url or not content:
            return None, None, "Không tìm thấy đủ Text hoặc Ảnh chuẩn trên trang này."
            
        img_response = requests.get(image_url, headers=headers)
        image = Image.open(BytesIO(img_response.content)).convert("RGB")
        
        full_text = f"{title}. {content}"
        return image, full_text, title
    except Exception as e:
        return None, None, f"Lỗi cào dữ liệu: {e}"

# ==========================================
# 4. HÀM DỰ ĐOÁN CHÍNH
# ==========================================
def predict_news(url):
    print(f"\n🌍 Đang truy cập URL: {url}")
    image, full_text, title = scrape_any_news_url(url)
    
    if image is None:
        print(f"❌ THẤT BẠI: {title}")
        return
        
    print(f"📰 Tiêu đề bài báo: {title}")
    
    # Ép kiểu dữ liệu
    img_tensor = val_transform(image).unsqueeze(0).to(device)
    txt_tensor, lens_tensor = process_text_for_prediction(full_text)
    txt_tensor, lens_tensor = txt_tensor.to(device), lens_tensor.to(device)
    
    # Tiên đoán
    with torch.no_grad():
        outputs = model(img_tensor, txt_tensor, lens_tensor)
        probabilities = F.softmax(outputs, dim=1).squeeze()
        
    # Lấy Top 3 kết quả cao nhất
    top3_prob, top3_idx = torch.topk(probabilities, 3)
    
    print("\n🔮 KẾT QUẢ DỰ ĐOÁN TỪ AI:")
    print("-" * 40)
    for i in range(3):
        cat_name = id_to_category[top3_idx[i].item()]
        conf = top3_prob[i].item() * 100
        if i == 0:
            print(f"🏆 Top 1: [{cat_name}] - Độ tự tin: {conf:.2f}%")
        else:
            print(f"   Top {i+1}: [{cat_name}] - Độ tự tin: {conf:.2f}%")
    print("-" * 40)

# ==========================================
# CHẠY THỬ NGHIỆM TẠI ĐÂY
# ==========================================
# Bạn có thể thay đường link này bằng bất kỳ bài báo nào bạn muốn thử!
test_urls = [
    "https://eva.vn/tin-tuc/con-dac-san-lan-ngup-day-dac-trong-be-nuoi-thanh-cong-anh-nong-dan-nghe-an-noi-co-tung-nao-cung-ban-het-c73a664584.html"
]

for url in test_urls:
    predict_news(url)

🤖 Đang khởi động AI trên thiết bị: CUDA


RuntimeError: Error(s) in loading state_dict for MultimodalNewsClassifier:
	Missing key(s) in state_dict: "image_extractor.features.0.weight", "image_extractor.features.0.bias", "image_extractor.features.1.weight", "image_extractor.features.1.bias", "image_extractor.features.1.running_mean", "image_extractor.features.1.running_var", "image_extractor.features.4.weight", "image_extractor.features.4.bias", "image_extractor.features.5.weight", "image_extractor.features.5.bias", "image_extractor.features.5.running_mean", "image_extractor.features.5.running_var", "image_extractor.features.8.weight", "image_extractor.features.8.bias", "image_extractor.features.9.weight", "image_extractor.features.9.bias", "image_extractor.features.9.running_mean", "image_extractor.features.9.running_var", "image_extractor.features.12.weight", "image_extractor.features.12.bias", "image_extractor.features.13.weight", "image_extractor.features.13.bias", "image_extractor.features.13.running_mean", "image_extracto